# Cache Traces With Ground-Truth Drift\n\nThis artifact provides standardized key-access-trace datasets for evaluating **cache admission policies** under popularity skew and popularity drift:\n\n- `real_twitter_cache_trace` — a sample (cluster026, 80,000 requests) of Twitter's production in-memory caching (Twemcache/Pelikan) traces, released alongside Yang et al., \"The CacheLib Caching Engine\", OSDI 2020.\n- `synthetic_zipf_alpha08/10/12` — 850,000-request synthetic traces over a 20,000-key universe following a Zipf rank-frequency law (alpha in {0.8, 1.0, 1.2}), with **injected ground-truth drift**: periodic rank-reshuffle events and randomly-timed cold-key popularity bursts. Every row's drift-event membership is embedded in `metadata_drift_event`.\n\nThe original `data.py` script standardizes the 4 raw per-trace JSON files into the `exp_sel_data_out` schema (one example per request row) and writes mini/preview/full-split output files. This notebook reproduces that same standardization logic — unchanged — on a small demo slice of one trace, then visualizes the resulting key-popularity distribution."


In [ ]:
import subprocess, sys\ndef _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])\n\n# loguru -- NOT pre-installed on Colab, always install\n_pip('loguru==0.7.3')\n\n# matplotlib -- pre-installed on Colab, install locally only (to match Colab's exact version)\nif 'google.colab' not in sys.modules:\n    _pip('matplotlib==3.10.0')


In [ ]:
import json\nimport sys\nfrom pathlib import Path\n\nfrom loguru import logger\nimport matplotlib.pyplot as plt\nfrom collections import Counter\n\nlogger.remove()\nlogger.add(sys.stdout, level="INFO", format="{time:HH:mm:ss}|{level:<7}|{message}")


## Load the demo data\n\n`mini_demo_data.json` holds 100 raw request rows sampled from the `synthetic_zipf_alpha10` trace, in the exact raw shape that `data.py` reads from `temp/datasets/full_synthetic_zipf_alpha10.json` before standardization (`{input: {...}, output, metadata_fold, metadata: {...}}` per row)."


In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/ai-inventor-papers/ai-invention-b940ce-shadow-queue-admission-with-recency/main/round-1/dataset-1/demo/mini_demo_data.json"\nimport json, os\n\ndef load_data():\n    try:\n        import urllib.request\n        with urllib.request.urlopen(GITHUB_DATA_URL) as response:\n            return json.loads(response.read().decode())\n    except Exception: pass\n    if os.path.exists("mini_demo_data.json"):\n        with open("mini_demo_data.json") as f: return json.load(f)\n    raise FileNotFoundError("Could not load mini_demo_data.json")


In [ ]:
data = load_data()\nprint(f"loaded {len(data)} raw rows")\nprint(data[0])


## Config\n\nThe original `data.py` splits the full standardized output into <100MB parts (`TARGET_PART_BYTES = 90_000_000`) and takes the first 3 examples per dataset for the mini/preview files (`n_mini_examples = 3`). Here we scale `TARGET_PART_BYTES` way down so the demo's tiny 100-row sample actually exercises the splitting-into-parts branch of the code, and expose `n_mini_examples` / `n_preview_examples` as config so they can be scaled back up to the original `3` (or higher) once you have real full-size data locally."


In [ ]:
# TARGET_PART_BYTES = 90_000_000  # original data.py value (keeps each split part under the 100MB GitHub cap)\nTARGET_PART_BYTES = 4_000  # scaled WAY down so our 100-row demo sample still splits into multiple parts\n\nn_mini_examples = 3       # original data.py value: first 3 examples per dataset for mini_data_out.json\nn_preview_examples = 3    # original data.py value: first 3 examples per dataset for preview_data_out.json (also truncates strings)\ntrunc_len = 200            # original data.py value: preview string truncation length\n\ndataset_name = "synthetic_zipf_alpha10"  # which of the 4 datasets this demo slice was sampled from


## Standardize rows into the `exp_sel_data_out` schema\n\n`row_to_example` (copied unchanged from `data.py`) turns one raw trace row into one standardized example: `input`/`output` become JSON-string / plain-string (schema requirement), and everything else flattens into `metadata_*` keys."


In [ ]:
def row_to_example(row: dict) -> dict:\n    """One trace row -> one exp_sel_data_out example. input/output are strings\n    (schema requirement); all other fields flatten into metadata_* keys."""\n    inp = row["input"]\n    meta = row["metadata"]\n    example = {\n        "input": json.dumps(\n            {\n                "seq": inp["seq"],\n                "timestamp": inp["timestamp"],\n                "key": inp["key"],\n                "trace_id": inp["trace_id"],\n                "request_type": inp["request_type"],\n            }\n        ),\n        "output": str(row["output"]),\n        "metadata_fold": row["metadata_fold"],\n        "metadata_seq": inp["seq"],\n        "metadata_key": inp["key"],\n        "metadata_trace_id": inp["trace_id"],\n        "metadata_request_type": inp["request_type"],\n        "metadata_source": meta["source"],\n        "metadata_drift_event": meta["drift_event"],\n        "metadata_alpha": meta["alpha"],\n        "metadata_trace_name": meta["trace_name"],\n    }\n    # extra real-trace-only fields (key_size, value_size, client_id, ttl, provenance)\n    for extra_key in ("key_size", "value_size", "client_id", "ttl", "provenance"):\n        if extra_key in meta:\n            example[f"metadata_{extra_key}"] = meta[extra_key]\n    return example\n\n\nexamples = [row_to_example(r) for r in data]\nlogger.info(f"{dataset_name}: {len(examples)} examples")\nexamples[0]


## Build the mini/preview outputs\n\nSame `trunc` truncation helper and mini/preview construction as `main()` in `data.py`, using `n_mini_examples` / `n_preview_examples` / `trunc_len` from the config cell instead of the hardcoded `3` / `200`."


In [ ]:
meta = {\n    "source": "twitter/cache-trace (real, OSDI'20 CacheLib) + synthetic Zipf-with-drift generator",\n    "description": "Cache access traces (real + synthetic-with-ground-truth-drift) for cache admission policy experiments",\n}\nout_datasets = [{"dataset": dataset_name, "examples": examples}]\ntotal = sum(len(d["examples"]) for d in out_datasets)\n\n\ndef trunc(o):\n    if isinstance(o, str) and len(o) > trunc_len:\n        return o[:trunc_len]\n    if isinstance(o, dict):\n        return {k: trunc(v) for k, v in o.items()}\n    if isinstance(o, list):\n        return [trunc(v) for v in o]\n    return o\n\n\nmini = {"metadata": meta, "datasets": [{"dataset": d["dataset"], "examples": d["examples"][:n_mini_examples]} for d in out_datasets]}\npreview = {\n    "metadata": meta,\n    "datasets": [{"dataset": d["dataset"], "examples": [trunc(e) for e in d["examples"][:n_preview_examples]]} for d in out_datasets],\n}\nlogger.info(f"mini: {sum(len(d['examples']) for d in mini['datasets'])} examples, preview: {sum(len(d['examples']) for d in preview['datasets'])} examples")\npreview


## Split the full standardized output into <100MB parts\n\nSame per-dataset splitting logic as `main()` in `data.py` — estimates bytes-per-example from a sample, then chunks so each part stays under `TARGET_PART_BYTES`. Writes into `demo_full_data_out/` under the current directory (instead of the original `WS / \"full_data_out\"`) and records a `_manifest.json` mapping dataset name -> ordered part filenames, exactly like the original."


In [ ]:
split_dir = Path("demo_full_data_out")\nsplit_dir.mkdir(exist_ok=True)\nfor f in split_dir.glob("full_data_out_*.json"):\n    f.unlink()\npart_idx = 1\nmanifest: dict[str, list[str]] = {}\nfor d in out_datasets:\n    name, ex = d["dataset"], d["examples"]\n    sample_n = min(200, len(ex))\n    bytes_per_example = len(json.dumps(ex[:sample_n])) / sample_n\n    chunk_n = max(1, int(TARGET_PART_BYTES / bytes_per_example))\n    manifest[name] = []\n    for i in range(0, len(ex), chunk_n):\n        part = ex[i : i + chunk_n]\n        part_fname = f"full_data_out_{part_idx}.json"\n        (split_dir / part_fname).write_text(\n            json.dumps({"metadata": meta, "datasets": [{"dataset": name, "examples": part}]})\n        )\n        manifest[name].append(part_fname)\n        part_idx += 1\n(split_dir / "_manifest.json").write_text(json.dumps(manifest, indent=2))\n\nlogger.info(f"saved {total} total examples across {part_idx - 1} full-data parts + mini/preview")\nmanifest


## Results: key-popularity distribution\n\nA quick summary table plus a plot of request counts per key, sorted by rank — this is the Zipf-skewed popularity curve that makes cache **admission policy** (deciding which keys are worth caching) a non-trivial problem, and the reason ground-truth drift events are injected: as ranks reshuffle, a policy tuned to one popularity ordering must adapt to the next."


In [ ]:
key_counts = Counter(e["metadata_key"] for e in examples)\nn_drift_rows = sum(1 for e in examples if e["metadata_drift_event"] is not None)\n\nprint(f"{'dataset':<28}{dataset_name}")\nprint(f"{'total requests':<28}{len(examples)}")\nprint(f"{'unique keys':<28}{len(key_counts)}")\nprint(f"{'requests in a drift event':<28}{n_drift_rows}")\nprint(f"{'alpha':<28}{examples[0]['metadata_alpha']}")\nprint(f"{'most requested key':<28}{key_counts.most_common(1)[0]}")\n\nranked_counts = [c for _, c in key_counts.most_common()]\nplt.figure(figsize=(6, 4))\nplt.bar(range(1, len(ranked_counts) + 1), ranked_counts, color="#4C72B0")\nplt.xlabel("key rank (most -> least popular)")\nplt.ylabel("request count")\nplt.title(f"Key-popularity distribution ({dataset_name}, n={len(examples)} requests)")\nplt.tight_layout()\nplt.show()
